# **Installation and Licensing**

In [ ]:
!pip install -q gamspy # Installs the GAMSPy package. GAMSPy documentation: https://gamspy.readthedocs.io/en/latest/

In [ ]:
!gamspy show license

In [ ]:
!gamspy list solvers

In [ ]:
!gamspy install license <path_to_ascii_file or access code> # paste your license here (if demo-license is not enough)

In [ ]:
# Optional: Install all solvers
!gamspy install solver --install-all-solvers

In [ ]:
!gamspy list solvers # Lists all available solvers in the gamspy package

# **Mathematical Model**

**Sector-Constrained Diversified Portfolio Optimization**

\begin{align}
    \min \quad & \sum_{i=1}^n \sum_{j=1}^n Q_{ij} x_i x_j - \lambda \sum_{i=1}^n \log(x_i + \varepsilon) \\
    \text{s.t.} \quad
    & \sum_{i=1}^n x_i = 1 \quad && \text{(Budget)} \\
    & x_i \leq y_i, \quad \forall i \quad && \text{(Linking)} \\
    & \sum_{i=1}^n y_i \leq 5 \quad && \text{(Cardinality)} \\
    & \sum_{s=1}^S z_s \leq 3 \quad && \text{(Sector limit)} \\
    & y_i \leq \sum_{s=1}^S \text{sector\_indicator}_{i,s} \cdot z_s \quad && \forall i \quad \text{(Sector-asset link)} \\
    & \text{sector\_weight}_s = \sum_{i=1}^n \text{sector\_indicator}_{i,s} \cdot x_i \quad && \forall s \quad \text{(Aggregation)} \\
    & \text{sector\_weight}_s \leq \text{sector\_max\_exposure}_s \cdot z_s \quad && \forall s \quad \text{(Exposure)} \\
    & x_i \geq 0, \quad y_i, z_s \in \{0, 1\} \quad && \text{(Domains)}
\end{align}

$$
\begin{array}{l l l r r r r}
\text{Name} & \text{Type} & \text{C} & \#\text{Vars} & \#\text{BinVars} & \#\text{IntVars} & \#\text{Cons} \\
\hline
 SaCCDPOP5\_3\_17 & MBNLP & Convex & 50 & 25 & 0 & 53 \\
SaCCDPOP5\_3\_61 & MBNLP & Convex & 132 & 66 & 0 & 135 \\
SaCCDPOP5\_3\_115 & MBNLP & Convex & 240 & 120 & 0 & 243 \\
SaCCDPOP5\_3\_326 & MBNLP & Convex & 662 & 331 & 0 & 665 \\
\end{array}
$$


#  **Solver Setup**


## SaCCDPOP5\_3\_17

In [ ]:
from gamspy import Container, Set, Parameter, Variable, Equation, Model, Sum, Sense, Alias
import numpy as np
import yfinance as yf
from gamspy.math import log, sqrt, power

# --- Data prep ---
tickers = [
    "XOM","DIS","INTU","LOW","MDT","BA","JPM","KO",
    "X","NKE","CAT","DDOG","NET","ROKU","AAPL","GOOGL","TSLA",
    "ROKU","ETSY","SHOP", "MELI"
]

data = yf.download(tickers, start='2023-01-01', end='2024-05-31')['Close']
returns = data.pct_change().dropna()
cov = returns.cov().values
n_assets = len(cov)
assets = [f"asset_{i+1}" for i in range(n_assets)]

sector_names = [f"sector_{i+1}" for i in range(5)]
n_sectors = len(sector_names)

# Calculate how many assets per sector (evenly divide)
assets_per_sector = n_assets // n_sectors
remainder = n_assets % n_sectors

sector_map = []

for s in range(n_sectors):
    count = assets_per_sector + (1 if s < remainder else 0)  # distribute remainder
    sector_map.extend([sector_names[s]] * count)

# Ensure sector_map length == n_assets (just in case)
sector_map = sector_map[:n_assets]

# Create dict mapping assets to sectors
sector_map_dict = dict(zip(assets, sector_map))

# --- GAMSPy model ---
m = Container()
i = Set(m, name="i", records=assets)
j = Alias(m, name="j", alias_with=i)
s = Set(m, name="s", records=sector_names)

# Sector mapping as a 2D parameter
sector_indicator = Parameter(m, name="sector_indicator", domain=[i, s], records=[
    (a, s_, 1.0 if sector_map_dict[a] == s_ else 0.0)
    for a in assets for s_ in sector_names
])

# Parameters
Q = Parameter(m, name="Q", domain=[i, j],
              records=[(assets[r], assets[c], cov[r, c])
                       for r in range(n_assets) for c in range(n_assets)])

sector_max_exposure = Parameter(m, name="sector_max_exposure", domain=[s], records=[
    (s_, 0.6) for s_ in sector_names
])
C = 5   # Chose C assets ...
S = 3   # from S sectors

# Variables
x = Variable(m, name="x", domain=[i], type="Positive")
y = Variable(m, name="y", domain=[i], type="Binary")
z = Variable(m, name="z", domain=[s], type="Binary")
sector_weight = Variable(m, name="sector_weight", domain=[s], type="Positive")

# --- Constraints ---

# Budget constraint
budget = Equation(m, name="budget")
budget[...] = Sum(i, x[i]) == 1

# Link constraint
link = Equation(m, name="link", domain=[i])
link[...] = x[i] <= y[i]

# Asset and sector constraint
# sector_asset_limit = Equation(m, name="sector_asset_limit")
# sector_asset_limit[...] = Sum(s, z[s]) <= Sum(i, y[i])

# Cardinality constraint
cardinality = Equation(m, name="cardinality")
cardinality[...] = Sum(i, y[i]) <= C

# Limit number of sectors used
sector_limit = Equation(m, name="sector_limit")
sector_limit[...] = Sum(s, z[s]) <= S

# Sector and investing link constraint
sector_link = Equation(m, name="sector_link", domain=[i])
sector_link[i] = y[i] <= Sum(s, sector_indicator[i, s] * z[s])

# Sector aggregation using the indicator
sector_aggregation = Equation(m, name="sector_aggregation", domain=[s])
sector_aggregation[s] = sector_weight[s] == Sum(i, sector_indicator[i, s] * x[i])

# Sector exposure constraint
sector_exposure = Equation(m, name="sector_exposure", domain=[s])
sector_exposure[s] = sector_weight[s] <= sector_max_exposure[s] * z[s]

# Combined objective: minimize variance + small weight penalty
lambda_div = 0.01  # diversification encouragement
epsilon = 1e-3  # small positive constant (needed for the dual problem - feasible)

portfolio_variance = Sum([i, j], x[i] * Q[i, j] * x[j])
diversification_penalty = Sum(i, log(x[i] + epsilon))
objective_expr = portfolio_variance - lambda_div * diversification_penalty


# --- Model ---
model = Model(m,
              name="SaCCDPOP_5_3_17",
              equations=m.getEquations(),
              problem="MINLP",
              sense=Sense.MIN,
              objective=objective_expr)

# --- Solve ---
import sys
model.solve(output=sys.stdout, solver="SHOT", solver_options={"Dual.MIP.Solver": 2})
    #"Model.Convexity.AssumeConvex": True

In [ ]:
# Convert GAMSPy model to GAMS (Note: One needs to download both the .gms and .gdx file)
model.toGams(path="gams")

After this, the files were opened and combined in GAMS Studio by GAMS Convert. Below is the rest of the codes for this type of problem.


## SaCCDPOP5\_3\_61

In [ ]:
from gamspy import Container, Set, Parameter, Variable, Equation, Model, Sum, Sense, Alias
import numpy as np
import yfinance as yf
from gamspy.math import log, sqrt, power

# --- Data prep ---
tickers = [
    "XOM","DIS","INTU","LOW","MDT","BA","JPM","KO",
    "X","NKE","CAT","DDOG","NET","ROKU","AAPL","GOOGL","TSLA",
    "ROKU","ETSY","SHOP", "MELI", "BA", "CAT", "CVX", "DIS", "JNJ", "KO", "MCD", "MRK", "NKE", "PFE",
    "CRM", "IBM", "INTU", "LOW", "MDT", "MMM", "RTX", "TMO", "UNP", "ZTS",
    "AAPL",
    "GOOGL",
    "TSLA",
    "NVDA",
    "AMZN",
    "META",
    "MSFT",
    "AMD",
    "PLTR",
    "SOFI",
    "LMT",
    "PZZA",
    "INTC",
    "PFE",
    "BA",
    "JPM",
    "KO",
    "XOM",
    "DIS",
    "NFLX",
    "SNAP",
    "UBER",
    "ZM",
    "SHOP",
    "DOCU",
    "BABA",
    "NVAX",
    "MRNA",
    "CRWD",
    "OKTA",
    "FSLY",
    "DDOG",
    "NET",
    "ROKU",
    "ETSY",
    "MELI",
    "JD",
    "BYND",
    "SBUX",
    "NKE",
    "CAT",
    "GE",
    "GM",
    "F",
    "TM",
    "V"
]

data = yf.download(tickers, start='2023-01-01', end='2024-05-31')['Close']
returns = data.pct_change().dropna()
cov = returns.cov().values
n_assets = len(cov)
assets = [f"asset_{i+1}" for i in range(n_assets)]

sector_names = [f"sector_{i+1}" for i in range(5)]
n_sectors = len(sector_names)

# Calculate how many assets per sector (evenly divide)
assets_per_sector = n_assets // n_sectors
remainder = n_assets % n_sectors

sector_map = []

for s in range(n_sectors):
    count = assets_per_sector + (1 if s < remainder else 0)  # distribute remainder
    sector_map.extend([sector_names[s]] * count)

# Ensure sector_map length == n_assets (just in case)
sector_map = sector_map[:n_assets]

# Create dict mapping assets to sectors
sector_map_dict = dict(zip(assets, sector_map))

# --- GAMSPy model ---
m = Container()
i = Set(m, name="i", records=assets)
j = Alias(m, name="j", alias_with=i)
s = Set(m, name="s", records=sector_names)

# Sector mapping as a 2D parameter
sector_indicator = Parameter(m, name="sector_indicator", domain=[i, s], records=[
    (a, s_, 1.0 if sector_map_dict[a] == s_ else 0.0)
    for a in assets for s_ in sector_names
])

# Parameters
Q = Parameter(m, name="Q", domain=[i, j],
              records=[(assets[r], assets[c], cov[r, c])
                       for r in range(n_assets) for c in range(n_assets)])

sector_max_exposure = Parameter(m, name="sector_max_exposure", domain=[s], records=[
    (s_, 0.6) for s_ in sector_names
])
C = 5   # Chose C assets ...
S = 3   # from S sectors

# Variables
x = Variable(m, name="x", domain=[i], type="Positive")
y = Variable(m, name="y", domain=[i], type="Binary")
z = Variable(m, name="z", domain=[s], type="Binary")
sector_weight = Variable(m, name="sector_weight", domain=[s], type="Positive")

# --- Constraints ---

# Budget constraint
budget = Equation(m, name="budget")
budget[...] = Sum(i, x[i]) == 1

# Link constraint
link = Equation(m, name="link", domain=[i])
link[...] = x[i] <= y[i]

# Asset and sector constraint
# sector_asset_limit = Equation(m, name="sector_asset_limit")
# sector_asset_limit[...] = Sum(s, z[s]) <= Sum(i, y[i])

# Cardinality constraint
cardinality = Equation(m, name="cardinality")
cardinality[...] = Sum(i, y[i]) <= C

# Limit number of sectors used
sector_limit = Equation(m, name="sector_limit")
sector_limit[...] = Sum(s, z[s]) <= S

# Sector and investing link constraint
sector_link = Equation(m, name="sector_link", domain=[i])
sector_link[i] = y[i] <= Sum(s, sector_indicator[i, s] * z[s])

# Sector aggregation using the indicator
sector_aggregation = Equation(m, name="sector_aggregation", domain=[s])
sector_aggregation[s] = sector_weight[s] == Sum(i, sector_indicator[i, s] * x[i])

# Sector exposure constraint
sector_exposure = Equation(m, name="sector_exposure", domain=[s])
sector_exposure[s] = sector_weight[s] <= sector_max_exposure[s] * z[s]

# Combined objective: minimize variance + small weight penalty
lambda_div = 0.01  # diversification encouragement
epsilon = 1e-3  # small positive constant (needed for the dual problem - feasible)

portfolio_variance = Sum([i, j], x[i] * Q[i, j] * x[j])
diversification_penalty = Sum(i, log(x[i] + epsilon))
objective_expr = portfolio_variance - lambda_div * diversification_penalty


# --- Model ---
model = Model(m,
              name="SaCCDPOP_5_3_61",
              equations=m.getEquations(),
              problem="MINLP",
              sense=Sense.MIN,
              objective=objective_expr)

# --- Solve ---
import sys
model.solve(output=sys.stdout, solver="SHOT", solver_options={"Dual.MIP.Solver": 2})
    #"Model.Convexity.AssumeConvex": True

## SaCCDPOP5\_3\_115

In [ ]:
from gamspy import Container, Set, Parameter, Variable, Equation, Model, Sum, Sense, Alias
import numpy as np
import yfinance as yf
from gamspy.math import log, sqrt, power

# --- Data prep ---
tickers = [
    "XOM","DIS","INTU","LOW","MDT","BA","JPM","KO",
    "X","NKE","CAT","DDOG","NET","ROKU","AAPL","GOOGL","TSLA",
    "ROKU","ETSY","SHOP", "MELI", "BA", "CAT", "CVX", "DIS", "JNJ", "KO", "MCD", "MRK", "NKE", "PFE",
    "CRM", "IBM", "INTU", "LOW", "MDT", "MMM", "RTX", "TMO", "UNP", "ZTS",
    "AAPL",
    "GOOGL",
    "TSLA",
    "NVDA",
    "AMZN",
    "META",
    "MSFT",
    "AMD",
    "PLTR",
    "SOFI",
    "LMT",
    "PZZA",
    "INTC",
    "PFE",
    "BA",
    "JPM",
    "KO",
    "XOM",
    "DIS",
    "NFLX",
    "SNAP",
    "UBER",
    "ZM",
    "SHOP",
    "DOCU",
    "BABA",
    "NVAX",
    "MRNA",
    "CRWD",
    "OKTA",
    "FSLY",
    "DDOG",
    "NET",
    "ROKU",
    "ETSY",
    "MELI",
    "JD",
    "BYND",
    "SBUX",
    "NKE",
    "CAT",
    "GE",
    "GM",
    "F",
    "TM",
    "V",
    "PEP", "NFLX", "TXN", "QCOM", "AVGO", "AMD", "SBUX", "GILD",
    "BA", "CAT", "CVX", "DIS", "JNJ", "KO", "MCD", "MRK", "NKE", "PFE",
    "PG", "T", "VZ", "WMT", "XOM", "GS", "JPM", "UNH", "HD", "ORCL",
    "CRM", "IBM", "INTU", "LOW", "MDT", "MMM", "RTX", "TMO", "UNP", "ZTS",
    "AMGN", "BLK", "BKNG", "COST", "DE", "DHR", "EL", "FDX", "GE", "GM",
    "HON", "IBM", "ISRG", "JNJ", "LMT", "LIN", "LRCX", "MAR", "MDLZ", "MET",
    "MO", "MS", "NEE", "NOC", "NVDA", "PYPL", "SBUX", "SCHW", "SO", "SPGI",
    "TGT", "TMUS", "TXN", "UPS", "USB", "V", "VLO", "VZ", "WBA", "WFC",
    "XEL", "ZBRA", "ZBH", "SNAP", "DOCU", "UBER", "ZM", "SHOP",
     "ADP", "BMY", "C",  "DOW",  "EMR",  "F",
    "GM",  "HCA",   "LVS",  "PYPL", "BK"
]

data = yf.download(tickers, start='2023-01-01', end='2024-05-31')['Close']
returns = data.pct_change().dropna()
cov = returns.cov().values
n_assets = len(cov)
assets = [f"asset_{i+1}" for i in range(n_assets)]

sector_names = [f"sector_{i+1}" for i in range(5)]
n_sectors = len(sector_names)

# Calculate how many assets per sector (evenly divide)
assets_per_sector = n_assets // n_sectors
remainder = n_assets % n_sectors

sector_map = []

for s in range(n_sectors):
    count = assets_per_sector + (1 if s < remainder else 0)  # distribute remainder
    sector_map.extend([sector_names[s]] * count)

# Ensure sector_map length == n_assets (just in case)
sector_map = sector_map[:n_assets]

# Create dict mapping assets to sectors
sector_map_dict = dict(zip(assets, sector_map))

# --- GAMSPy model ---
m = Container()
i = Set(m, name="i", records=assets)
j = Alias(m, name="j", alias_with=i)
s = Set(m, name="s", records=sector_names)

# Sector mapping as a 2D parameter
sector_indicator = Parameter(m, name="sector_indicator", domain=[i, s], records=[
    (a, s_, 1.0 if sector_map_dict[a] == s_ else 0.0)
    for a in assets for s_ in sector_names
])

# Parameters
Q = Parameter(m, name="Q", domain=[i, j],
              records=[(assets[r], assets[c], cov[r, c])
                       for r in range(n_assets) for c in range(n_assets)])

sector_max_exposure = Parameter(m, name="sector_max_exposure", domain=[s], records=[
    (s_, 0.6) for s_ in sector_names
])
C = 5   # Chose C assets ...
S = 3   # from S sectors

# Variables
x = Variable(m, name="x", domain=[i], type="Positive")
y = Variable(m, name="y", domain=[i], type="Binary")
z = Variable(m, name="z", domain=[s], type="Binary")
sector_weight = Variable(m, name="sector_weight", domain=[s], type="Positive")

# --- Constraints ---

# Budget constraint
budget = Equation(m, name="budget")
budget[...] = Sum(i, x[i]) == 1

# Link constraint
link = Equation(m, name="link", domain=[i])
link[...] = x[i] <= y[i]

# Asset and sector constraint
# sector_asset_limit = Equation(m, name="sector_asset_limit")
# sector_asset_limit[...] = Sum(s, z[s]) <= Sum(i, y[i])

# Cardinality constraint
cardinality = Equation(m, name="cardinality")
cardinality[...] = Sum(i, y[i]) <= C

# Limit number of sectors used
sector_limit = Equation(m, name="sector_limit")
sector_limit[...] = Sum(s, z[s]) <= S

# Sector and investing link constraint
sector_link = Equation(m, name="sector_link", domain=[i])
sector_link[i] = y[i] <= Sum(s, sector_indicator[i, s] * z[s])

# Sector aggregation using the indicator
sector_aggregation = Equation(m, name="sector_aggregation", domain=[s])
sector_aggregation[s] = sector_weight[s] == Sum(i, sector_indicator[i, s] * x[i])

# Sector exposure constraint
sector_exposure = Equation(m, name="sector_exposure", domain=[s])
sector_exposure[s] = sector_weight[s] <= sector_max_exposure[s] * z[s]

# Combined objective: minimize variance + small weight penalty
lambda_div = 0.01  # diversification encouragement
epsilon = 1e-3  # small positive constant (needed for the dual problem - feasible)

portfolio_variance = Sum([i, j], x[i] * Q[i, j] * x[j])
diversification_penalty = Sum(i, log(x[i] + epsilon))
objective_expr = portfolio_variance - lambda_div * diversification_penalty


# --- Model ---
model = Model(m,
              name="SaCCDPOP_5_3_115",
              equations=m.getEquations(),
              problem="MINLP",
              sense=Sense.MIN,
              objective=objective_expr)

# --- Solve ---
import sys
model.solve(output=sys.stdout, solver="SHOT", solver_options={"Dual.MIP.Solver": 2})
    #"Model.Convexity.AssumeConvex": True

## SaCCDPOP5\_3\_326


In [ ]:
from gamspy import Container, Set, Parameter, Variable, Equation, Model, Sum, Sense, Alias
import numpy as np
import yfinance as yf
from gamspy.math import log, sqrt, power

# --- Data prep ---
tickers = [
    "XOM","DIS","INTU","LOW","MDT","BA","JPM","KO",
    "X","NKE","CAT","DDOG","NET","ROKU","AAPL","GOOGL","TSLA",
    "ROKU","ETSY","SHOP", "MELI", "BA", "CAT", "CVX", "DIS", "JNJ", "KO", "MCD", "MRK", "NKE", "PFE",
    "CRM", "IBM", "INTU", "LOW", "MDT", "MMM", "RTX", "TMO", "UNP", "ZTS",
    "AAPL",
    "GOOGL",
    "TSLA",
    "NVDA",
    "AMZN",
    "META",
    "MSFT",
    "AMD",
    "PLTR",
    "SOFI",
    "LMT",
    "PZZA",
    "INTC",
    "PFE",
    "BA",
    "JPM",
    "KO",
    "XOM",
    "DIS",
    "NFLX",
    "SNAP",
    "UBER",
    "ZM",
    "SHOP",
    "DOCU",
    "BABA",
    "NVAX",
    "MRNA",
    "CRWD",
    "OKTA",
    "FSLY",
    "DDOG",
    "NET",
    "ROKU",
    "ETSY",
    "MELI",
    "JD",
    "BYND",
    "SBUX",
    "NKE",
    "CAT",
    "GE",
    "GM",
    "F",
    "TM",
    "V",
    "PEP", "NFLX", "TXN", "QCOM", "AVGO", "AMD", "SBUX", "GILD",
    "BA", "CAT", "CVX", "DIS", "JNJ", "KO", "MCD", "MRK", "NKE", "PFE",
    "PG", "T", "VZ", "WMT", "XOM", "GS", "JPM", "UNH", "HD", "ORCL",
    "CRM", "IBM", "INTU", "LOW", "MDT", "MMM", "RTX", "TMO", "UNP", "ZTS",
    "AMGN", "BLK", "BKNG", "COST", "DE", "DHR", "EL", "FDX", "GE", "GM",
    "HON", "IBM", "ISRG", "JNJ", "LMT", "LIN", "LRCX", "MAR", "MDLZ", "MET",
    "MO", "MS", "NEE", "NOC", "NVDA", "PYPL", "SBUX", "SCHW", "SO", "SPGI",
    "TGT", "TMUS", "TXN", "UPS", "USB", "V", "VLO", "VZ", "WBA", "WFC",
    "XEL", "ZBRA", "ZBH", "SNAP", "DOCU", "UBER", "ZM", "SHOP",
     "ADP", "BMY", "C",  "DOW",  "EMR",  "F",
    "GM",  "HCA",   "LVS",  "PYPL", "BK",
    "XOM", "GS", "JPM", "UNH", "HD", "ORCL",
    "CRM", "IBM", "INTU", "LOW", "MDT", "MMM", "RTX", "TMO", "UNP", "ZTS",
    "AMGN", "BLK", "DHR", "EL", "FDX", "GE", "GM",
    "IBM", "ISRG", "LMT", "LIN", "LRCX", "MAR", "MDLZ", "MET",
    "MO", "MS", "NEE", "NOC", "NVDA", "PYPL", "SBUX", "SCHW", "SO", "SPGI",
    "TGT", "TMUS", "TXN", "UPS", "USB", "V", "VLO", "WBA", "WFC",
    "XEL", "ZBRA", "ZBH", "SNAP", "DOCU", "UBER", "ZM", "SHOP",
     "ADP", "BMY", "C",  "DOW",  "EMR",  "F",
    "GM",  "HCA",   "LVS",  "PYPL", "BK",
    "AAPL", "MSFT", "AMZN", "NVDA", "GOOGL", "META", "AVGO", "TSLA",
    "XOM", "UNH", "LLY", "PG", "MA", "HD", "MRK", "CVX",
    "PEP", "ABBV", "COST", "BAC", "ADBE", "KO", "WMT", "CRM", "NFLX", "TMO",
    "ABT", "ACN", "PFE", "CSCO", "MCD", "DHR", "LIN", "INTC", "QCOM", "NEE",
    "WFC", "TXN", "PM", "MS", "BMY", "AMGN", "ORCL", "UNP", "AVB",
    "LOW", "UPS", "RTX", "AMD", "GS", "SPGI", "CVS", "SBUX", "BLK", "MDT",
    "ISRG", "LMT", "TGT", "BKNG", "GILD",
    "MMC", "ADP", "VRTX", "CB", "ELV", "REGN", "PNC", "FI", "C", "ADI",
    "CSX", "MDLZ", "SCHW", "PGR", "MU", "DUK", "EQIX", "SO", "HCA",
    "BDX", "ICE", "CL", "ITW", "APD", "SHW", "EOG", "TRV", "FDX", "HUM",
    "NKE", "PSA", "EMR", "ETN", "TFC", "GM", "AON", "WM", "ROP", "AIG",
    "MAR", "USB", "MCO", "D", "EW", "IDXX", "AEP", "FTNT", "ADSK",
    "ROST", "MNST", "KDP", "CMG", "MET", "CTAS", "ORLY", "PCAR", "CTSH", "AZO",
    "SPG", "WTW", "CME", "DLR", "PH", "F", "VLO", "PRU", "OXY", "MSI",
    "ALL", "TEL", "HLT", "HSY", "PEG",
    "EBAY", "DHI", "HES", "MCK", "COF", "KR", "WBD", "ED",
    "DVN", "SBAC", "OTIS", "EFX", "CHTR", "BKR", "DFS", "KEYS", "WMB", "VRSK",
    "FTV", "STT", "PPG", "CEG",  "CTVA",
    "PAYX", "BIIB", "XEL", "LHX", "MTD", "DRI", "ZBH", "KLAC", "TSCO", "TDG",
    "RMD", "PCG", "DOV", "VICI", "CBRE", "CMS", "EXC", "DLTR",
    "ATO", "CDW", "HIG", "LDOS", "CNP", "MLM", "LRCX", "AEE", "EIX",
    "RCL", "FANG", "PPL", "FAST", "IFF", "COR", "ETR", "TRGP", "MAA",
    "AKAM", "TSN", "EPAM", "NUE", "WEC", "HPE", "IR", "AIZ", "MKC", "RSG",
    "CAH", "VTR", "CHD", "BALL", "NDAQ", "NI", "AVY", "CRL", "DTE",
    "SJM", "FE", "WAT", "TTWO", "EXR", "BBY", "HOLX", "SWKS", "JKHY", "LKQ",
    "FMC", "TER", "NRG", "TYL", "PHM", "JBHT",
    "RL", "BR", "ZBRA", "BXP", "GNRC", "VMC", "STE", "ESS", "APA",
    "INCY", "TECH", "PKG", "CF", "HBAN", "AFL", "NVR", "ALB", "TXT", "DXCM",
    "KIM", "UHS", "BWA", "MOS", "BEN", "IP", "NDSN", "HII", "ARE", "PWR",
    "WHR", "LNT", "EMN", "TAP", "AMCR", "XRAY", "KMX", "L", "CINF", "SNA",
    "FRT", "NOV", "CAG", "GRMN", "WRB", "PARA", "ALLE", "AOS",
    "LKQ", "IPG", "QRVO", "WY", "GNW", "TXT", "PFG"
]

data = yf.download(tickers, start='2023-01-01', end='2024-05-31')['Close']
returns = data.pct_change().dropna()
cov = returns.cov().values
n_assets = len(cov)
assets = [f"asset_{i+1}" for i in range(n_assets)]

sector_names = [f"sector_{i+1}" for i in range(5)]
n_sectors = len(sector_names)

# Calculate how many assets per sector (evenly divide)
assets_per_sector = n_assets // n_sectors
remainder = n_assets % n_sectors

sector_map = []

for s in range(n_sectors):
    count = assets_per_sector + (1 if s < remainder else 0)  # distribute remainder
    sector_map.extend([sector_names[s]] * count)

# Ensure sector_map length == n_assets (just in case)
sector_map = sector_map[:n_assets]

# Create dict mapping assets to sectors
sector_map_dict = dict(zip(assets, sector_map))

# --- GAMSPy model ---
m = Container()
i = Set(m, name="i", records=assets)
j = Alias(m, name="j", alias_with=i)
s = Set(m, name="s", records=sector_names)

# Sector mapping as a 2D parameter
sector_indicator = Parameter(m, name="sector_indicator", domain=[i, s], records=[
    (a, s_, 1.0 if sector_map_dict[a] == s_ else 0.0)
    for a in assets for s_ in sector_names
])

# Parameters
Q = Parameter(m, name="Q", domain=[i, j],
              records=[(assets[r], assets[c], cov[r, c])
                       for r in range(n_assets) for c in range(n_assets)])

sector_max_exposure = Parameter(m, name="sector_max_exposure", domain=[s], records=[
    (s_, 0.6) for s_ in sector_names
])
C = 5   # Chose C assets ...
S = 3   # from S sectors

# Variables
x = Variable(m, name="x", domain=[i], type="Positive")
y = Variable(m, name="y", domain=[i], type="Binary")
z = Variable(m, name="z", domain=[s], type="Binary")
sector_weight = Variable(m, name="sector_weight", domain=[s], type="Positive")

# --- Constraints ---

# Budget constraint
budget = Equation(m, name="budget")
budget[...] = Sum(i, x[i]) == 1

# Link constraint
link = Equation(m, name="link", domain=[i])
link[...] = x[i] <= y[i]

# Asset and sector constraint
# sector_asset_limit = Equation(m, name="sector_asset_limit")
# sector_asset_limit[...] = Sum(s, z[s]) <= Sum(i, y[i])

# Cardinality constraint
cardinality = Equation(m, name="cardinality")
cardinality[...] = Sum(i, y[i]) <= C

# Limit number of sectors used
sector_limit = Equation(m, name="sector_limit")
sector_limit[...] = Sum(s, z[s]) <= S

# Sector and investing link constraint
sector_link = Equation(m, name="sector_link", domain=[i])
sector_link[i] = y[i] <= Sum(s, sector_indicator[i, s] * z[s])

# Sector aggregation using the indicator
sector_aggregation = Equation(m, name="sector_aggregation", domain=[s])
sector_aggregation[s] = sector_weight[s] == Sum(i, sector_indicator[i, s] * x[i])

# Sector exposure constraint
sector_exposure = Equation(m, name="sector_exposure", domain=[s])
sector_exposure[s] = sector_weight[s] <= sector_max_exposure[s] * z[s]

# Combined objective: minimize variance + small weight penalty
lambda_div = 0.01  # diversification encouragement
epsilon = 1e-3  # small positive constant (needed for the dual problem - feasible)

portfolio_variance = Sum([i, j], x[i] * Q[i, j] * x[j])
diversification_penalty = Sum(i, log(x[i] + epsilon))
objective_expr = portfolio_variance - lambda_div * diversification_penalty


# --- Model ---
model = Model(m,
              name="SaCCDPOP_5_3_326e",
              equations=m.getEquations(),
              problem="MINLP",
              sense=Sense.MIN,
              objective=objective_expr)

# --- Solve ---
import sys
model.solve(output=sys.stdout, solver="SHOT", solver_options={"Dual.MIP.Solver": 0})
    #"Model.Convexity.AssumeConvex": True